In [1]:

!pip install -q langchain langchain-ollama llama-index llama-index-llms-ollama llama-index-embeddings-huggingface

import subprocess
import time

# Start Ollama server
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(5)

# Download the language model
!ollama pull llama3.2:1b

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# Configure LlamaIndex
Settings.llm = Ollama(model="llama3.2:1b", request_timeout=120)
Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

# Sample legal document
legal_text = """
This agreement is between ABC Company and XYZ Services.
The agreement starts on 1 January 2026 and lasts for one year.
XYZ Services must protect confidential company information.
Either party may terminate the agreement by giving 30 days written notice.
Disputes will be resolved through arbitration.
"""

# Create and index the document
documents = [Document(text=legal_text)]
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()

# Set up LangChain
llm = ChatOllama(model="llama3.2:1b", temperature=0)

summary_prompt = ChatPromptTemplate.from_template("""
Summarize this legal document in simple language.
Mention the parties, duration, obligations, and termination conditions.
Use only the provided text.

Document:
{context}
""")

qa_prompt = ChatPromptTemplate.from_template("""
Answer the question using only the legal document context.
If the answer is missing, say it is not specified in the document.

Context:
{context}

Question: {question}
Answer:
""")

summary_chain = summary_prompt | llm
qa_chain = qa_prompt | llm

# Summarize the legal document
print("LEGAL DOCUMENT SUMMARY")
summary = summary_chain.invoke({"context": legal_text})
print(summary.content)

# Retrieve information by asking a question
question = "How can either party terminate the agreement?"
context = query_engine.query(question)

print("\nLEGAL QUESTION:", question)
answer = qa_chain.invoke({
    "context": str(context),
    "question": question
})
print("ANSWER:", answer.content)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

LEGAL DOCUMENT SUMMARY
Here is a summary of the document in simple language:

**Agreement Details:**

* The agreement is between two companies: ABC Company and XYZ Services.
* It starts on January 1, 2026, and lasts for one year.
* XYZ Services must keep confidential any information about ABC Company.
* Either party can end the agreement by giving 30 days' written notice.
* Disputes will be resolved through arbitration.

**Key Points:**

* Confidentiality: XYZ Services must protect ABC Company's information.
* Termination: Either party can end the agreement after 30 days' notice.
* Dispute Resolution: Disputes will be resolved through arbitration.

LEGAL QUESTION: How can either party terminate the agreement?
ANSWER: The answer is: by giving the other party 30 days written notice.
